# Qwen3-4B -- self-distillation on teacher-failed samples (dual-GPU)

Tests whether STaNR's self-distillation step (which improved Gemma3-4B and
plain Qwen3-4B, but *hurt* qwen3-4b-thinking -- see `STaNR_en_v1_*.json`)
also degrades plain Qwen3-4B when the self-distilled samples come from
Qwen3-4B's own checkpoint rather than a different model. If it does, that
confirms the effect is a self-loop / confirmation-bias problem specific to
distilling a model from itself, not something specific to the "thinking"
architecture.

**Pipeline**:
1. Load the already-trained `qwen3-4b-sft-w-reasoning-eng-finqa-vinumqa`
   checkpoint (SFT w ENG reasoning trace, PA-match-only).
2. Identify the 870 `train.json` samples the teacher (gemma-4-31B-it)
   could **not** verify (`train.json` minus
   `train_with_reasoning_trace_en_pa_distiil_gemma.json`, matched by `id`).
3. Run this Qwen3-4B checkpoint on those 870 samples (dual-GPU, same
   inference setup as `qwen3-4b-eval-only-dual-gpu.ipynb`), keeping both the
   reasoning trace and the generated program per sample.
4. Filter to only the samples where the generated program matches gold
   under PA (`scorer.equal_program`) -- same filter STaNR_v1 used.
5. Merge the PA-verified self-distilled samples into the base PA-match-only
   training set, in the same `qa.reasoning_trace` / `qa.trace_source =
   "self_distilled"` shape as `STaNR_en_v1_train.json`, and save as a new
   dataset ready for a fresh SFT run.

No training happens in this notebook -- only self-distillation + dataset
construction. Fine-tune on the output file separately.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


### Config

In [ ]:
MAX_SEQ_LENGTH = 4567  # ViNumQA contexts (pre_text + table + post_text) can be long
ADAPTER_DIR = "/kaggle/input/datasets/ntphuc/qwen3-4b-sft-w-reasoning-eng-finqa-vinumqa/qwen3-4b-vinumqa-sft-adapter"

# Adjust these two to wherever the base ViNumQA files and the base PA-match-only
# distilled training set live in this Kaggle input.
TRAIN_JSON_PATH = "/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/train.json"
TRAIN_PA_EN_PATH = "/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/train_with_reasoning_trace_en_pa_distiil_gemma.json"

OUTPUT_PATH = "/kaggle/working/STaNR_en_v1_qwen3-4b_self_train.json"


### Find the teacher-failed samples

`train.json` minus `train_with_reasoning_trace_en_pa_distiil_gemma.json`,
matched by `id` -- these are exactly the 870 samples gemma-4-31B-it could
not solve well enough to pass the PA filter during the original
independent-solve distillation.

In [ ]:
import json

train_full = json.load(open(TRAIN_JSON_PATH, encoding="utf-8"))
train_verified = json.load(open(TRAIN_PA_EN_PATH, encoding="utf-8"))

verified_ids = {x["id"] for x in train_verified}
teacher_failed = [x for x in train_full if x["id"] not in verified_ids]

print(f"train.json total: {len(train_full)}")
print(f"teacher-verified (PA match only, EN): {len(train_verified)}")
print(f"teacher-failed (to self-distill): {len(teacher_failed)}")


### Format teacher-failed samples for inference

Same context formatting as every other prompting/SFT notebook in this repo,
plus `id` kept through the pipeline (needed to merge results back into the
base training set later).

In [ ]:
import pandas as pd
from tabulate import tabulate

def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["id", "pre_text_processed", "table_processed", "table_raw", "post_text_processed",
             "input_question", "program_processed", "answer_processed"]]
    df.columns = ["id", "pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
    return df

failed_df = pd.DataFrame(teacher_failed)
failed_df = process_split(failed_df)
failed_df.sample(n=3)


In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so the checkpoint
# sees exactly the same instruction shape it was trained on.


### Dual-GPU self-distillation

Splits `failed_df` across Kaggle's two T4s. Each GPU gets its own OS process
with its own model copy (loaded fresh inside the worker) -- CUDA contexts
don't fork safely, and `multiprocessing`'s `spawn` start method can't pickle
a function defined inline in a notebook cell, so the worker is written out
as a standalone script and launched via `subprocess` instead.

Not using vLLM: found unreliable on Kaggle T4s in an earlier session (bf16
unsupported on compute capability 7.5, OOM from VRAM not fully released
across engine restarts within the same process).

In [ ]:
worker_script = r'''
import argparse
import gc
import json
import os
import sys

parser = argparse.ArgumentParser()
parser.add_argument("--gpu", type=int, required=True)
parser.add_argument("--input_json", type=str, required=True)
parser.add_argument("--output_json", type=str, required=True)
parser.add_argument("--adapter_dir", type=str, required=True)
parser.add_argument("--max_seq_length", type=int, required=True)
args = parser.parse_args()

os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)  # must be set before importing torch/unsloth

import torch
from unsloth import FastLanguageModel

SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=args.adapter_dir,
    max_seq_length=args.max_seq_length,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
FastLanguageModel.for_inference(model)

rows = json.load(open(args.input_json, encoding="utf-8"))
results = {}
n_total = len(rows)

for i, row in enumerate(rows, start=1):
    prompt = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, max_new_tokens=1024,
            temperature=0.6, top_p=0.95, top_k=20, min_p=0,
        )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    try:
        idx = len(output_ids) - output_ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        idx = 0

    # Self-distillation needs both halves of the generation: the reasoning
    # trace (everything before </think>) to fill qa.reasoning_trace in the
    # new training sample, and the program (after </think>) to check against
    # gold via PA before deciding whether to keep the sample at all.
    trace_text = tokenizer.decode(output_ids[:idx], skip_special_tokens=True).strip()
    program_text = tokenizer.decode(output_ids[idx:], skip_special_tokens=True).strip()

    results[str(row["index"])] = {
        "id": row["id"],
        "gold_program": row["gold_program"],
        "gold_answer": row["gold_answer"],
        "reasoning_trace": trace_text,
        "generated_program": program_text,
    }

    print(f"{i}/{n_total}", flush=True)

    gc.collect()
    torch.cuda.empty_cache()

with open(args.output_json, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False)

print(f"[gpu {args.gpu}] done: {len(results)} samples", flush=True)
'''

with open("/kaggle/working/_dual_gpu_worker.py", "w", encoding="utf-8") as f:
    f.write(worker_script)

print("Worker script written to /kaggle/working/_dual_gpu_worker.py")


In [ ]:
import re
import subprocess
import threading
import queue

_PROGRESS_LINE_RE = re.compile(r"^\d+/\d+$")  # matches worker's "i/n_total" progress lines only

half = len(failed_df) // 2
chunks = [failed_df.iloc[:half], failed_df.iloc[half:]]

input_paths = []
output_paths = []
for i, chunk in enumerate(chunks):
    rows = [
        {
            "index": idx,
            "id": row["id"],
            "pre_text": row["pre_text"], "table": row["table"],
            "post_text": row["post_text"], "question": row["question"],
            "gold_program": row["program"], "gold_answer": row["answer"],
        }
        for idx, row in chunk.iterrows()
    ]
    in_path = f"/kaggle/working/_selfdistill_chunk_{i}_input.json"
    out_path = f"/kaggle/working/_selfdistill_chunk_{i}_output.json"
    with open(in_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)
    input_paths.append(in_path)
    output_paths.append(out_path)

procs = []
for gpu_id, in_path, out_path in zip([0, 1], input_paths, output_paths):
    cmd = [
        "python", "/kaggle/working/_dual_gpu_worker.py",
        "--gpu", str(gpu_id),
        "--input_json", in_path,
        "--output_json", out_path,
        "--adapter_dir", ADAPTER_DIR,
        "--max_seq_length", str(MAX_SEQ_LENGTH),
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    procs.append(proc)

log_queue = queue.Queue()

def _reader(gpu_id, proc):
    for line in proc.stdout:
        log_queue.put((gpu_id, line.rstrip()))
    log_queue.put((gpu_id, None))  # sentinel: this process's stream is closed (process exited)

threads = [
    threading.Thread(target=_reader, args=(gpu_id, proc), daemon=True)
    for gpu_id, proc in zip([0, 1], procs)
]
for t in threads:
    t.start()

n_total_all = len(failed_df)
n_done_all = 0
n_finished_streams = 0

while n_finished_streams < len(procs):
    gpu_id, line = log_queue.get()
    if line is None:
        n_finished_streams += 1
        continue
    if _PROGRESS_LINE_RE.match(line):
        n_done_all += 1
        print(f"TOTAL: {n_done_all}/{n_total_all}")
    else:
        print(f"[gpu {gpu_id}] {line}")  # errors, the final "done: N samples" still show

for proc in procs:
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Worker process failed with exit code {proc.returncode}")

self_distill_results = {}
for out_path in output_paths:
    with open(out_path, encoding="utf-8") as f:
        self_distill_results.update(json.load(f))

print(f"Done. {len(self_distill_results)} / {len(failed_df)} samples generated.")


### PA filter + merge into the base training set

Same scorer as every other prompting/SFT notebook in this repo
(`notebooks/evaluate/scorer.py`), inlined here so this notebook has no
external file dependency on Kaggle. Only self-distilled samples whose
program matches gold under PA are kept -- same filter STaNR_v1 used.

In [ ]:
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented. See notebooks/evaluate/scorer.py
in this repo for the full docstring covering the five corrections applied.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


def str_to_num(text: str) -> Union[float, str]:
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps


def program_tokenization(original_program: str) -> List[str]:
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching \')\' found) in program: \'{original_program}\'"
            )

        program.append(m.group(1) + "(")
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: \'{text[pos:]}\' (from: \'{original_program}\')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text


def equal_program(program1: List[str], program2: List[str]) -> bool:
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


In [ ]:
# Only keep samples where the self-distilled program matches gold (PA) --
# same filter STaNR_v1 used, via equal_program/program_tokenization.
verified_self_distilled = []
for idx_str, r in self_distill_results.items():
    try:
        pred_tok = program_tokenization(extract_program(r["generated_program"]))
        gold_tok = program_tokenization(r["gold_program"])
        is_match = equal_program(gold_tok, pred_tok)
    except Exception:
        is_match = False
    if is_match:
        verified_self_distilled.append(r)

print(f"Self-distilled PA-verified: {len(verified_self_distilled)} / {len(self_distill_results)}")

# Build qa.reasoning_trace / qa.trace_source entries, matching the id -> full
# sample lookup from the original teacher-failed rows, so pre_text/table/
# post_text/id all come from train.json unchanged -- exactly like STaNR_v1.
id_to_sample = {x["id"]: x for x in teacher_failed}

new_samples = []
for r in verified_self_distilled:
    sample = dict(id_to_sample[r["id"]])  # shallow copy
    sample["qa"] = dict(sample["qa"])
    sample["qa"]["reasoning_trace"] = r["reasoning_trace"]
    sample["qa"]["trace_source"] = "self_distilled"
    new_samples.append(sample)

stanr_qwen3_4b_self = train_verified + new_samples
print(f"STaNR (self-distilled from Qwen3-4B itself): {len(stanr_qwen3_4b_self)} total "
      f"({len(train_verified)} base + {len(new_samples)} self-distilled)")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(stanr_qwen3_4b_self, f, ensure_ascii=False)

print(f"Saved to {OUTPUT_PATH}")


### Next step

Fine-tune a fresh Qwen3-4B (starting from the base model, not this
checkpoint) on `STaNR_en_v1_qwen3-4b_self_train.json`, using the same SFT
setup as `vsf-qwen3-4b-sft-w-reasoning-eng-finqa-vinumqa.ipynb`, then eval
on `test.json` and compare PA/EA against:
- SFT (w ENG reasoning trace; distill - PA match only): PA 0.6519 / EA 0.6962
- STaNR_v1 (cross-model, self-distilled from qwen3-4b-thinking): PA 0.6559 / EA 0.7123

If this run (self-distilled from Qwen3-4B's own checkpoint) also degrades
relative to the PA-match-only baseline the way qwen3-4b-thinking did, that
confirms the self-loop hypothesis rather than something specific to the
thinking architecture.